# Lecture 02: LLM Architecture — MLP

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/wusche1/Illiad_ML_Engineering/blob/main/lectures/01_a_ml_foundations/exercises/04_mlp/notebook.ipynb)

In [ ]:
import os, importlib
if os.getenv('COLAB_RELEASE_TAG'):
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/wusche1/Illiad_ML_Engineering/main/lectures/01_a_ml_foundations/exercises/04_mlp/utils.py",
        "utils.py"
    )
    importlib.invalidate_caches()

import utils
importlib.reload(utils)
from utils import (
    gelu_new,
    test_gelu, test_mlp, test_mlp_shapes, test_residual_mlp,
)

In [ ]:
import torch
import torch.nn as nn
import einops
import matplotlib.pyplot as plt

## The MLP in a Transformer

The MLP (multi-layer perceptron) is one of two core components in each transformer block. While attention moves information between positions, the MLP processes each position **independently and identically**.

It's a simple two-layer neural network with a nonlinear activation in between:

$$\text{MLP}(x) = \text{GELU}(x W_{\text{in}} + b_{\text{in}}) \, W_{\text{out}} + b_{\text{out}}$$

The hidden dimension is typically $d_{\text{mlp}} = 4 \times d_{\text{model}}$.

<img src="https://raw.githubusercontent.com/callummcdougall/computational-thread-art/master/example_images/misc/transformer-mlp-new-2.png" width="600">

Intuition: once attention has gathered relevant information to a position, the MLP can do the actual computation, reasoning, and lookup. **What exactly happens inside MLPs** is a major open problem in mechanistic interpretability.

## Exercise A: GELU Activation

Transformers use GELU (Gaussian Error Linear Unit) instead of ReLU. GPT-2 uses an approximation:

$$\text{GELU}(x) = 0.5 \, x \left(1 + \tanh\!\left(\sqrt{\frac{2}{\pi}} \left(x + 0.044715 \, x^3\right)\right)\right)$$

GELU is smooth everywhere (unlike ReLU's kink at 0). For large positive $x$ it's approximately the identity; for large negative $x$ it's approximately zero.

In [ ]:
def my_gelu(x):
    # TODO (~1 line): implement the GELU approximation formula above
    pass

In [ ]:
test_gelu(my_gelu)

<details>
<summary><b>Solution</b></summary>

```python
def my_gelu(x):
    return 0.5 * x * (1.0 + torch.tanh((2.0 / torch.pi) ** 0.5 * (x + 0.044715 * x ** 3)))
```
</details>

In [ ]:
# Compare GELU vs ReLU
x = torch.linspace(-3, 3, 200)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(x, gelu_new(x), label='GELU', linewidth=2)
ax.plot(x, torch.relu(x), label='ReLU', linewidth=2, linestyle='--')
ax.axhline(0, color='gray', linewidth=0.5)
ax.axvline(0, color='gray', linewidth=0.5)
ax.set(xlabel='x', ylabel='f(x)', title='GELU vs ReLU')
ax.legend()
plt.tight_layout()
plt.show()

---

## Exercise B: Implement an MLP Layer

Implement the MLP as an `nn.Module`. It maps from `d_model` to `d_mlp` (with GELU), then back to `d_model`.

Use raw `nn.Parameter` weight matrices (not `nn.Linear`), so you see every matrix multiply explicitly.

The shapes:
- `W_in`: `(d_model, d_mlp)`, `b_in`: `(d_mlp,)`
- `W_out`: `(d_mlp, d_model)`, `b_out`: `(d_model,)`

You can use `einops.einsum` or `@` for the matrix multiplies.

In [ ]:
class MLP(nn.Module):
    def __init__(self, d_model=64, d_mlp=256):
        super().__init__()
        self.W_in = nn.Parameter(torch.randn(d_model, d_mlp) * 0.02)
        self.b_in = nn.Parameter(torch.zeros(d_mlp))
        self.W_out = nn.Parameter(torch.randn(d_mlp, d_model) * 0.02)
        self.b_out = nn.Parameter(torch.zeros(d_model))

    def forward(self, x):
        """
        Args:
            x: (batch, seq, d_model)
        Returns:
            (batch, seq, d_model)
        """
        # TODO (~3 lines): project up, apply GELU, project back down
        pass

In [ ]:
test_mlp_shapes(MLP)
test_mlp(MLP)

<details>
<summary><b>Hint</b></summary>

The forward pass is: `x @ W_in + b_in`, then `gelu_new(...)`, then `result @ W_out + b_out`.
</details>

<details>
<summary><b>Solution</b></summary>

```python
class MLP(nn.Module):
    def __init__(self, d_model=64, d_mlp=256):
        super().__init__()
        self.W_in = nn.Parameter(torch.randn(d_model, d_mlp) * 0.02)
        self.b_in = nn.Parameter(torch.zeros(d_mlp))
        self.W_out = nn.Parameter(torch.randn(d_mlp, d_model) * 0.02)
        self.b_out = nn.Parameter(torch.zeros(d_model))

    def forward(self, x):
        pre = x @ self.W_in + self.b_in
        post = gelu_new(pre)
        return post @ self.W_out + self.b_out
```
</details>

---

## Exercise C: Residual MLP Stack

In a transformer, MLP outputs are **added back** to the input via a residual connection:

$$x_{\ell+1} = x_\ell + \text{MLP}(x_\ell)$$

This is crucial. Without residual connections, deep networks suffer from vanishing gradients and the signal degrades as it passes through many layers. With them, the gradient can flow directly from the output back to any layer.

The residual stream is the central object of a transformer. Each layer reads from it and adds to it. Think of it as a shared memory that every layer can access.

Build a stack of `n_layers` MLP layers, each with a residual connection.

In [ ]:
class ResidualMLP(nn.Module):
    def __init__(self, d_model=64, d_mlp=256, n_layers=3):
        super().__init__()
        self.layers = nn.ModuleList([MLP(d_model, d_mlp) for _ in range(n_layers)])

    def forward(self, x):
        """
        Args:
            x: (batch, seq, d_model)
        Returns:
            (batch, seq, d_model)
        """
        # TODO (~2 lines): pass x through each layer with a residual connection
        pass

In [ ]:
test_residual_mlp(ResidualMLP)

<details>
<summary><b>Hint</b></summary>

Loop over `self.layers` and do `x = x + layer(x)` each time.
</details>

<details>
<summary><b>Solution</b></summary>

```python
class ResidualMLP(nn.Module):
    def __init__(self, d_model=64, d_mlp=256, n_layers=3):
        super().__init__()
        self.layers = nn.ModuleList([MLP(d_model, d_mlp) for _ in range(n_layers)])

    def forward(self, x):
        for layer in self.layers:
            x = x + layer(x)
        return x
```
</details>

### Visualize the residual stream

Let's see what the residual stream looks like as it passes through layers. We track the norm at each layer to see how information accumulates.

In [ ]:
torch.manual_seed(0)
model = ResidualMLP(d_model=64, d_mlp=256, n_layers=6)
x = torch.randn(1, 20, 64)

norms = [x.norm(dim=-1).mean().item()]
residual = x
for layer in model.layers:
    residual = residual + layer(residual)
    norms.append(residual.norm(dim=-1).mean().item())

plt.figure(figsize=(8, 4))
plt.plot(norms, 'o-', linewidth=2)
plt.xlabel('Layer')
plt.ylabel('Mean residual stream norm')
plt.title('Residual stream grows as layers add to it')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()